# First experiment: Printing all 100 seeds with Nighthawk to visualize how often this happens

In [21]:
import os
import sys
import time
import gc
import numpy as np
from typing import Dict, List
from qiskit import transpile
from qiskit import qasm2
from qiskit.circuit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService

def load_target_qasm_circuits(circuits_dir: str, target_files: List[str]) -> Dict[str, QuantumCircuit]:
    """
    Parses a specific list of QASM files from a target directory into QuantumCircuit objects.
    I/O operations are strictly isolated here to prevent timing contamination.
    """
    if not os.path.isdir(circuits_dir):
        sys.exit(f"FATAL: Directory '{circuits_dir}' does not exist.")

    circuits = {}
    for filename in target_files:
        filepath = os.path.join(circuits_dir, filename)
        if not os.path.isfile(filepath):
            print(f"[!] WARNING: Target file '{filepath}' not found. Bypassing.")
            continue
            
        try:
            # Using qasm2.load to maintain legacy custom instruction support from your original spec
            circuits[filename] = qasm2.load(
                filepath,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                custom_classical=qasm2.LEGACY_CUSTOM_CLASSICAL,
            )
        except Exception as e:
            print(f"[!] WARNING: Parsing failed for '{filename}'. Error: {e}")
            
    if not circuits:
        sys.exit("FATAL: No valid circuits loaded from the provided list. Terminating.")
        
    return circuits

def profile_transpiler_disparity(
    circuits: Dict[str, QuantumCircuit],
    backend_name: str = "ibm_nighthawk",
    optimization_level: int = 3,
    number_seeds: int = 10,
    base_seed: int = 42
) -> Dict[str, dict]:
    """
    Profiles the runtime disparity of the Qiskit transpiler across different seeds.
    """
    print(f"[*] Authenticating with IBM Quantum Runtime Service for '{backend_name}'...")
    try:
        service = QiskitRuntimeService(channel="ibm_quantum_platform")
        backend = service.backend(backend_name)
    except Exception as e:
        sys.exit(f"FATAL: Backend initialization failed. Error: {e}")

    hw_qubits = backend.num_qubits
    print(f"[*] Target Architecture: {backend.name} | Logical Qubits: {hw_qubits}\n")

    seeds = [base_seed + i for i in range(number_seeds)]
    disparity_results = {}

    for circ_name, qc in circuits.items():
        logical_qubits = qc.num_qubits
        print(f"[>] Profiling: '{circ_name}' | Required Qubits: {logical_qubits}")
        
        if logical_qubits > hw_qubits:
            print(f"    [!] Violation: Circuit exceeds backend limits ({hw_qubits}). Bypassing.")
            continue

        runtimes: List[float] = []

        for seed in seeds:
            try:
                # Disable garbage collection to prevent unpredictable CPU halts during transpilation profiling
                gc.disable() 
                start_time = time.perf_counter()
                
                _ = transpile(
                    qc,
                    backend=backend,
                    optimization_level=optimization_level,
                    seed_transpiler=seed
                )
                
                exec_time = time.perf_counter() - start_time
                gc.enable() # Re-enable GC immediately after timing block
                
                runtimes.append(exec_time)
                print(f"    - Seed {seed:04d} | Runtime: {exec_time:.4f} sec")
                
            except Exception as e:
                gc.enable() # Ensure GC is re-enabled on failure
                print(f"    [!] FATAL: Transpilation failed on seed {seed}. Error: {e}")

        if runtimes:
            mean_time = np.mean(runtimes)
            std_time = np.std(runtimes)
            min_time = np.min(runtimes)
            max_time = np.max(runtimes)
            coeff_var = (std_time / mean_time) * 100 if mean_time > 0 else 0
            spread = max_time - min_time

            disparity_results[circ_name] = {
                "mean_sec": mean_time,
                "std_sec": std_time,
                "cv_percent": coeff_var
            }
            
            print(f"    === Runtime Disparity Metrics ===")
            print(f"    Mean   : {mean_time:.4f}s")
            print(f"    StdDev : {std_time:.4f}s")
            print(f"    Range  : [{min_time:.4f}s, {max_time:.4f}s]")
            print(f"    CV     : {coeff_var:.2f}%\n")
        else:
            print(f"    [!] No successful compilation runs for '{circ_name}'.\n")

    return disparity_results

if __name__ == "__main__":
    # Define your exact target files here.
    TARGET_QASM_FILES = [
        "adder_n64.qasm",
        "swap_test_n41.qasm",
        "swap_test_n83.qasm",
        "knn_n41.qasm",
        "knn_n67.qasm",
    ]
    
    TARGET_DIR = "circuits"
    
    # 1. Load only the explicitly requested files
    loaded_circuits = load_target_qasm_circuits(TARGET_DIR, TARGET_QASM_FILES)
    
    # 2. Execute the profiling pipeline
    profile_transpiler_disparity(
        circuits=loaded_circuits,
        backend_name="ibm_phoenix", 
        optimization_level=3,
        number_seeds=50, 
        base_seed=420
    )

[*] Authenticating with IBM Quantum Runtime Service for 'ibm_phoenix'...
[*] Target Architecture: ibm_phoenix | Logical Qubits: 120

[>] Profiling: 'adder_n64.qasm' | Required Qubits: 64
    - Seed 0420 | Runtime: 0.2615 sec
    - Seed 0421 | Runtime: 0.7465 sec
    - Seed 0422 | Runtime: 0.2364 sec
    - Seed 0423 | Runtime: 1.1133 sec
    - Seed 0424 | Runtime: 0.2371 sec
    - Seed 0425 | Runtime: 0.0643 sec
    - Seed 0426 | Runtime: 0.2280 sec
    - Seed 0427 | Runtime: 0.2216 sec
    - Seed 0428 | Runtime: 0.0757 sec
    - Seed 0429 | Runtime: 0.0597 sec
    - Seed 0430 | Runtime: 0.0583 sec
    - Seed 0431 | Runtime: 0.2198 sec
    - Seed 0432 | Runtime: 0.0686 sec
    - Seed 0433 | Runtime: 0.0609 sec
    - Seed 0434 | Runtime: 0.2549 sec
    - Seed 0435 | Runtime: 0.2367 sec
    - Seed 0436 | Runtime: 0.2113 sec
    - Seed 0437 | Runtime: 0.3846 sec
    - Seed 0438 | Runtime: 0.1048 sec
    - Seed 0439 | Runtime: 0.1709 sec
    - Seed 0440 | Runtime: 0.0484 sec
    - Seed 0441

### Really big gap between two seeds

## Second Experiment: Running pass by pass comparaison between multiple seeds to find the guilty pass

### NightHawk Ibm phoenix

In [12]:
import os
import sys
import time
import gc
import numpy as np
from collections import defaultdict
from typing import Dict, List

from qiskit import qasm2
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

def load_target_qasm_circuits(circuits_dir: str, target_files: List[str]) -> Dict[str, QuantumCircuit]:
    if not os.path.isdir(circuits_dir):
        sys.exit(f"FATAL: Directory '{circuits_dir}' does not exist.")

    circuits = {}
    for filename in target_files:
        filepath = os.path.join(circuits_dir, filename)
        if not os.path.isfile(filepath):
            print(f"[!] WARNING: Target file '{filepath}' not found. Bypassing.")
            continue
            
        try:
            circuits[filename] = qasm2.load(
                filepath,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                custom_classical=qasm2.LEGACY_CUSTOM_CLASSICAL,
            )
        except Exception as e:
            print(f"[!] WARNING: Parsing failed for '{filename}'. Error: {e}")
            
    if not circuits:
        sys.exit("FATAL: No valid circuits loaded. Terminating.")
        
    return circuits

def profile_pass_manager_disparity(
    circuits: Dict[str, QuantumCircuit],
    backend_name: str,
    optimization_level: int = 3,
    number_seeds: int = 10,
    base_seed: int = 42
):
    print(f"[*] Authenticating with IBM Quantum Runtime Service...")
    try:
        service = QiskitRuntimeService(channel="ibm_quantum_platform") 
        backend = service.backend(backend_name)
        target = backend.target
    except Exception as e:
        sys.exit(f"FATAL: Backend initialization failed. Error: {e}")

    hw_qubits = backend.num_qubits
    print(f"[*] Target Architecture: {backend.name} | Logical Qubits: {hw_qubits}\n")

    seeds = [base_seed + i for i in range(number_seeds)]

    for circ_name, qc in circuits.items():
        logical_qubits = qc.num_qubits
        print(f"{'='*90}")
        print(f"[>] Profiling Circuit: '{circ_name}' | Qubits: {logical_qubits}")
        
        if logical_qubits > hw_qubits:
            print(f"    [!] Violation: Exceeds backend limits. Bypassing.")
            continue

        pass_seed_times = defaultdict(list)
        total_runtimes = []

        for seed in seeds:
            pm = generate_preset_pass_manager(
                optimization_level=optimization_level,
                target=target,
                seed_transpiler=seed
            )
            
            current_seed_pass_accumulation = defaultdict(float)

            def universal_pass_callback(**kwargs):
                task_obj = kwargs.get('task') or kwargs.get('pass_')
                exec_time = kwargs.get('running_time') or kwargs.get('time')
                
                if task_obj is not None and exec_time is not None:
                    pass_name = type(task_obj).__name__
                    current_seed_pass_accumulation[pass_name] += exec_time

            try:
                gc.disable()
                t_start = time.perf_counter()
                
                _ = pm.run(qc, callback=universal_pass_callback)
                
                t_total = time.perf_counter() - t_start
                gc.enable()
                
                total_runtimes.append(t_total)
                
                for p_name, t_val in current_seed_pass_accumulation.items():
                    pass_seed_times[p_name].append(t_val)
                
                # Minimal progress indicator
                if (seed - base_seed + 1) % 10 == 0:
                    print(f"    ... Completed {seed - base_seed + 1}/{number_seeds} seeds")
                    
            except Exception as e:
                gc.enable()
                print(f"    [!] FATAL: Pipeline failed on seed {seed}. Error: {e}")
                continue

        if not total_runtimes:
            print(f"    [!] No successful runs for {circ_name}.")
            continue

        pass_metrics = []
        for p_name, times in pass_seed_times.items():
            if len(times) < number_seeds:
                # Pad with zeros if a pass did not execute in every seed
                times.extend([0.0] * (number_seeds - len(times)))
                
            p_mean = np.mean(times)
            p_std = np.std(times)
            p_max = np.max(times)
            p_min = np.min(times)
            p_gap = p_max - p_min  # Absolute maximum discrepancy between any two seeds
            p_cv = (p_std / p_mean * 100) if p_mean > 0 else 0
            
            pass_metrics.append({
                'name': p_name,
                'mean': p_mean,
                'std': p_std,
                'max': p_max,
                'min': p_min,
                'gap': p_gap,
                'cv': p_cv
            })

        # Sorted by Max Gap descending to satisfy the requirement
        pass_metrics.sort(key=lambda x: x['gap'], reverse=True)
        mean_total = np.mean(total_runtimes)
        std_total = np.std(total_runtimes)
        max_total = np.max(total_runtimes)
        min_total = np.min(total_runtimes)

        print(f"\n    [Total Pipeline Runtime Disparity - {circ_name}]")
        print(f"    Mean: {mean_total:.4f}s | StdDev: {std_total:.4f}s | Max Gap: {(max_total - min_total):.4f}s | CV: {(std_total/mean_total*100):.2f}%")
        
        print(f"\n    [Top 15 Passes by Seed Runtime Gap - {circ_name}]")
        print(f"    {'Pass Name'.ljust(32)} | {'Max Gap (s)'.ljust(12)} | {'StdDev (s)'.ljust(12)} | {'Mean (s)'.ljust(12)} | {'CV (%)'}")
        print(f"    {'-'*90}")
        
        for metric in pass_metrics[:15]:
            print(f"    {metric['name'].ljust(32)} | {metric['gap']:<12.4f} | {metric['std']:<12.4f} | {metric['mean']:<12.4f} | {metric['cv']:.1f}%")
        print("\n")


if __name__ == "__main__":
    TARGET_QASM_FILES = [
        "adder_n28.qasm",
        "adder_n64.qasm",
        "swap_test_n41.qasm",
        "swap_test_n83.qasm",
        "knn_n41.qasm",
        "knn_n67.qasm",
    ]
    
    TARGET_DIR = "circuits"
    
    loaded_circuits = load_target_qasm_circuits(TARGET_DIR, TARGET_QASM_FILES)
    
    profile_pass_manager_disparity(
        circuits=loaded_circuits,
        backend_name="ibm_phoenix", 
        optimization_level=3,
        number_seeds=100, 
        base_seed=420
    )

[*] Authenticating with IBM Quantum Runtime Service...
[*] Target Architecture: ibm_phoenix | Logical Qubits: 120

[>] Profiling Circuit: 'adder_n28.qasm' | Qubits: 28
    ... Completed 10/100 seeds
    ... Completed 20/100 seeds
    ... Completed 30/100 seeds
    ... Completed 40/100 seeds
    ... Completed 50/100 seeds
    ... Completed 60/100 seeds
    ... Completed 70/100 seeds
    ... Completed 80/100 seeds
    ... Completed 90/100 seeds
    ... Completed 100/100 seeds

    [Total Pipeline Runtime Disparity - adder_n28.qasm]
    Mean: 0.4612s | StdDev: 1.9766s | Max Gap: 14.6848s | CV: 428.61%

    [Top 15 Passes by Seed Runtime Gap - adder_n28.qasm]
    Pass Name                        | Max Gap (s)  | StdDev (s)   | Mean (s)     | CV (%)
    ------------------------------------------------------------------------------------------
    VF2PostLayout                    | 14.6810      | 1.9765       | 0.4479       | 441.2%
    TwoQubitPeepholeOptimization     | 0.0127       | 0.001

### Heron - IBM boston

In [13]:
import os
import sys
import time
import gc
import numpy as np
from collections import defaultdict
from typing import Dict, List

from qiskit import qasm2
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

def load_target_qasm_circuits(circuits_dir: str, target_files: List[str]) -> Dict[str, QuantumCircuit]:
    if not os.path.isdir(circuits_dir):
        sys.exit(f"FATAL: Directory '{circuits_dir}' does not exist.")

    circuits = {}
    for filename in target_files:
        filepath = os.path.join(circuits_dir, filename)
        if not os.path.isfile(filepath):
            print(f"[!] WARNING: Target file '{filepath}' not found. Bypassing.")
            continue
            
        try:
            circuits[filename] = qasm2.load(
                filepath,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                custom_classical=qasm2.LEGACY_CUSTOM_CLASSICAL,
            )
        except Exception as e:
            print(f"[!] WARNING: Parsing failed for '{filename}'. Error: {e}")
            
    if not circuits:
        sys.exit("FATAL: No valid circuits loaded. Terminating.")
        
    return circuits

def profile_pass_manager_disparity(
    circuits: Dict[str, QuantumCircuit],
    backend_name: str,
    optimization_level: int = 3,
    number_seeds: int = 10,
    base_seed: int = 42
):
    print(f"[*] Authenticating with IBM Quantum Runtime Service...")
    try:
        service = QiskitRuntimeService(channel="ibm_quantum_platform") 
        backend = service.backend(backend_name)
        target = backend.target
    except Exception as e:
        sys.exit(f"FATAL: Backend initialization failed. Error: {e}")

    hw_qubits = backend.num_qubits
    print(f"[*] Target Architecture: {backend.name} | Logical Qubits: {hw_qubits}\n")

    seeds = [base_seed + i for i in range(number_seeds)]

    for circ_name, qc in circuits.items():
        logical_qubits = qc.num_qubits
        print(f"{'='*90}")
        print(f"[>] Profiling Circuit: '{circ_name}' | Qubits: {logical_qubits}")
        
        if logical_qubits > hw_qubits:
            print(f"    [!] Violation: Exceeds backend limits. Bypassing.")
            continue

        pass_seed_times = defaultdict(list)
        total_runtimes = []

        for seed in seeds:
            pm = generate_preset_pass_manager(
                optimization_level=optimization_level,
                target=target,
                seed_transpiler=seed
            )
            
            current_seed_pass_accumulation = defaultdict(float)

            def universal_pass_callback(**kwargs):
                task_obj = kwargs.get('task') or kwargs.get('pass_')
                exec_time = kwargs.get('running_time') or kwargs.get('time')
                
                if task_obj is not None and exec_time is not None:
                    pass_name = type(task_obj).__name__
                    current_seed_pass_accumulation[pass_name] += exec_time

            try:
                gc.disable()
                t_start = time.perf_counter()
                
                _ = pm.run(qc, callback=universal_pass_callback)
                
                t_total = time.perf_counter() - t_start
                gc.enable()
                
                total_runtimes.append(t_total)
                
                for p_name, t_val in current_seed_pass_accumulation.items():
                    pass_seed_times[p_name].append(t_val)
                
                # Minimal progress indicator
                if (seed - base_seed + 1) % 10 == 0:
                    print(f"    ... Completed {seed - base_seed + 1}/{number_seeds} seeds")
                    
            except Exception as e:
                gc.enable()
                print(f"    [!] FATAL: Pipeline failed on seed {seed}. Error: {e}")
                continue

        if not total_runtimes:
            print(f"    [!] No successful runs for {circ_name}.")
            continue

        pass_metrics = []
        for p_name, times in pass_seed_times.items():
            if len(times) < number_seeds:
                # Pad with zeros if a pass did not execute in every seed
                times.extend([0.0] * (number_seeds - len(times)))
                
            p_mean = np.mean(times)
            p_std = np.std(times)
            p_max = np.max(times)
            p_min = np.min(times)
            p_gap = p_max - p_min  # Absolute maximum discrepancy between any two seeds
            p_cv = (p_std / p_mean * 100) if p_mean > 0 else 0
            
            pass_metrics.append({
                'name': p_name,
                'mean': p_mean,
                'std': p_std,
                'max': p_max,
                'min': p_min,
                'gap': p_gap,
                'cv': p_cv
            })

        # Sorted by Max Gap descending to satisfy the requirement
        pass_metrics.sort(key=lambda x: x['gap'], reverse=True)
        mean_total = np.mean(total_runtimes)
        std_total = np.std(total_runtimes)
        max_total = np.max(total_runtimes)
        min_total = np.min(total_runtimes)

        print(f"\n    [Total Pipeline Runtime Disparity - {circ_name}]")
        print(f"    Mean: {mean_total:.4f}s | StdDev: {std_total:.4f}s | Max Gap: {(max_total - min_total):.4f}s | CV: {(std_total/mean_total*100):.2f}%")
        
        print(f"\n    [Top 15 Passes by Seed Runtime Gap - {circ_name}]")
        print(f"    {'Pass Name'.ljust(32)} | {'Max Gap (s)'.ljust(12)} | {'StdDev (s)'.ljust(12)} | {'Mean (s)'.ljust(12)} | {'CV (%)'}")
        print(f"    {'-'*90}")
        
        for metric in pass_metrics[:15]:
            print(f"    {metric['name'].ljust(32)} | {metric['gap']:<12.4f} | {metric['std']:<12.4f} | {metric['mean']:<12.4f} | {metric['cv']:.1f}%")
        print("\n")


if __name__ == "__main__":
    TARGET_QASM_FILES = [
        "adder_n28.qasm",
        "adder_n64.qasm",
        "swap_test_n41.qasm",
        "swap_test_n83.qasm",
        "knn_n41.qasm",
        "knn_n67.qasm",
    ]
    
    TARGET_DIR = "circuits"
    
    loaded_circuits = load_target_qasm_circuits(TARGET_DIR, TARGET_QASM_FILES)
    
    profile_pass_manager_disparity(
        circuits=loaded_circuits,
        backend_name="ibm_boston", 
        optimization_level=3,
        number_seeds=100, 
        base_seed=420,
    )

[*] Authenticating with IBM Quantum Runtime Service...
[*] Target Architecture: ibm_boston | Logical Qubits: 156

[>] Profiling Circuit: 'adder_n28.qasm' | Qubits: 28
    ... Completed 10/100 seeds
    ... Completed 20/100 seeds
    ... Completed 30/100 seeds
    ... Completed 40/100 seeds
    ... Completed 50/100 seeds
    ... Completed 60/100 seeds
    ... Completed 70/100 seeds
    ... Completed 80/100 seeds
    ... Completed 90/100 seeds
    ... Completed 100/100 seeds

    [Total Pipeline Runtime Disparity - adder_n28.qasm]
    Mean: 0.0485s | StdDev: 0.0174s | Max Gap: 0.0713s | CV: 35.97%

    [Top 15 Passes by Seed Runtime Gap - adder_n28.qasm]
    Pass Name                        | Max Gap (s)  | StdDev (s)   | Mean (s)     | CV (%)
    ------------------------------------------------------------------------------------------
    VF2PostLayout                    | 0.0706       | 0.0178       | 0.0341       | 52.3%
    TwoQubitPeepholeOptimization     | 0.0142       | 0.0021   

### Eagle - FakeSherbrooke

In [14]:
import os
import sys
import time
import gc
from collections import defaultdict
from typing import Dict, List, Optional
import numpy as np

from qiskit import qasm2
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Robust import handling across Qiskit 1.0+ and legacy environments
try:
    from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
except ImportError:
    try:
        from qiskit.providers.fake_provider import FakeSherbrooke
    except ImportError:
        sys.exit(
            "FATAL: Unable to import FakeSherbrooke. Install qiskit-ibm-runtime "
            "(pip install qiskit-ibm-runtime) or ensure fake providers are available."
        )


def load_target_qasm_circuits(circuits_dir: str, target_files: List[str]) -> Dict[str, QuantumCircuit]:
    if not os.path.isdir(circuits_dir):
        sys.exit(f"FATAL: Directory '{circuits_dir}' does not exist.")

    circuits = {}
    for filename in target_files:
        filepath = os.path.join(circuits_dir, filename)
        if not os.path.isfile(filepath):
            print(f"[!] WARNING: Target file '{filepath}' not found. Bypassing.")
            continue

        try:
            circuits[filename] = qasm2.load(
                filepath,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                custom_classical=qasm2.LEGACY_CUSTOM_CLASSICAL,
            )
        except Exception as e:
            print(f"[!] WARNING: Parsing failed for '{filename}'. Error: {e}")

    if not circuits:
        sys.exit("FATAL: No valid circuits loaded. Terminating.")

    return circuits


def profile_pass_manager_disparity(
    circuits: Dict[str, QuantumCircuit],
    backend: Optional[FakeSherbrooke] = None,
    optimization_level: int = 3,
    number_seeds: int = 10,
    base_seed: int = 42,
):
    if backend is None:
        backend = FakeSherbrooke()

    target = backend.target
    hw_qubits = backend.num_qubits
    print(f"[*] Target Architecture: {backend.name} (Local Mock) | Physical Qubits: {hw_qubits}\n")

    seeds = [base_seed + i for i in range(number_seeds)]

    for circ_name, qc in circuits.items():
        logical_qubits = qc.num_qubits
        print(f"{'='*90}")
        print(f"[>] Profiling Circuit: '{circ_name}' | Qubits: {logical_qubits}")

        if logical_qubits > hw_qubits:
            print(f"    [!] Violation: Circuit requires {logical_qubits} qubits; backend has {hw_qubits}. Bypassing.")
            continue

        pass_seed_times = defaultdict(list)
        total_runtimes = []

        for seed in seeds:
            pm = generate_preset_pass_manager(
                optimization_level=optimization_level,
                target=target,
                seed_transpiler=seed,
            )

            current_seed_pass_accumulation = defaultdict(float)

            def universal_pass_callback(**kwargs):
                task_obj = kwargs.get("task") or kwargs.get("pass_")
                exec_time = kwargs.get("running_time") or kwargs.get("time")

                if task_obj is not None and exec_time is not None:
                    pass_name = type(task_obj).__name__
                    current_seed_pass_accumulation[pass_name] += exec_time

            # Reclaim unreferenced DAG objects before measuring to prevent cache/memory thrashing
            gc.collect()

            try:
                gc.disable()
                t_start = time.perf_counter()

                _ = pm.run(qc, callback=universal_pass_callback)

                t_total = time.perf_counter() - t_start
                gc.enable()

                total_runtimes.append(t_total)

                for p_name, t_val in current_seed_pass_accumulation.items():
                    pass_seed_times[p_name].append(t_val)

                if (seed - base_seed + 1) % 10 == 0:
                    print(f"    ... Completed {seed - base_seed + 1}/{number_seeds} seeds")

            except Exception as e:
                gc.enable()
                print(f"    [!] Pipeline failed on seed {seed}. Error: {e}")
                continue

        if not total_runtimes:
            print(f"    [!] No successful runs for {circ_name}.")
            continue

        pass_metrics = []
        for p_name, times in pass_seed_times.items():
            if len(times) < len(total_runtimes):
                # Account for passes conditionally invoked only under certain stochastic routing graphs
                times.extend([0.0] * (len(total_runtimes) - len(times)))

            p_mean = float(np.mean(times))
            p_std = float(np.std(times))
            p_max = float(np.max(times))
            p_min = float(np.min(times))
            p_gap = p_max - p_min
            p_cv = (p_std / p_mean * 100.0) if p_mean > 0 else 0.0

            pass_metrics.append({
                "name": p_name,
                "mean": p_mean,
                "std": p_std,
                "max": p_max,
                "min": p_min,
                "gap": p_gap,
                "cv": p_cv,
            })

        pass_metrics.sort(key=lambda x: x["gap"], reverse=True)

        mean_total = float(np.mean(total_runtimes))
        std_total = float(np.std(total_runtimes))
        max_total = float(np.max(total_runtimes))
        min_total = float(np.min(total_runtimes))

        print(f"\n    [Total Pipeline Runtime Disparity - {circ_name}]")
        print(
            f"    Mean: {mean_total:.4f}s | StdDev: {std_total:.4f}s | "
            f"Max Gap: {(max_total - min_total):.4f}s | CV: {(std_total/mean_total*100):.2f}%"
        )

        print(f"\n    [Top 15 Passes by Seed Runtime Gap - {circ_name}]")
        print(
            f"    {'Pass Name'.ljust(32)} | {'Max Gap (s)'.ljust(12)} | "
            f"{'StdDev (s)'.ljust(12)} | {'Mean (s)'.ljust(12)} | {'CV (%)'}"
        )
        print(f"    {'-'*90}")

        for metric in pass_metrics[:15]:
            print(
                f"    {metric['name'].ljust(32)} | {metric['gap']:<12.4f} | "
                f"{metric['std']:<12.4f} | {metric['mean']:<12.4f} | {metric['cv']:.1f}%"
            )
        print("\n")


if __name__ == "__main__":
    TARGET_QASM_FILES = [
        "adder_n28.qasm",
        "adder_n64.qasm",
        "swap_test_n41.qasm",
        "swap_test_n83.qasm",
        "knn_n41.qasm",
        "knn_n67.qasm",
    ]

    TARGET_DIR = "circuits"

    loaded_circuits = load_target_qasm_circuits(TARGET_DIR, TARGET_QASM_FILES)

    # FakeSherbrooke is a 127-qubit Eagle r3 mock backend
    backend_instance = FakeSherbrooke()

    profile_pass_manager_disparity(
        circuits=loaded_circuits,
        backend=backend_instance,
        optimization_level=3,
        number_seeds=100,
        base_seed=420,
    )

[*] Target Architecture: fake_sherbrooke (Local Mock) | Physical Qubits: 127

[>] Profiling Circuit: 'adder_n28.qasm' | Qubits: 28
    ... Completed 10/100 seeds
    ... Completed 20/100 seeds
    ... Completed 30/100 seeds
    ... Completed 40/100 seeds
    ... Completed 50/100 seeds
    ... Completed 60/100 seeds
    ... Completed 70/100 seeds
    ... Completed 80/100 seeds
    ... Completed 90/100 seeds
    ... Completed 100/100 seeds

    [Total Pipeline Runtime Disparity - adder_n28.qasm]
    Mean: 0.0286s | StdDev: 0.0352s | Max Gap: 0.3607s | CV: 123.15%

    [Top 15 Passes by Seed Runtime Gap - adder_n28.qasm]
    Pass Name                        | Max Gap (s)  | StdDev (s)   | Mean (s)     | CV (%)
    ------------------------------------------------------------------------------------------
    GateDirection                    | 0.3439       | 0.0342       | 0.0041       | 828.8%
    VF2PostLayout                    | 0.0171       | 0.0030       | 0.0054       | 55.5%
    Com

| Circuit | Qubits | Nighthawk Mean (s) | Nighthawk Max Gap (s) | Heron Mean (s) | Heron Max Gap (s) | Eagle Mean (s) | Eagle Max Gap (s) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| `adder_n28.qasm` | 28 | 0.4479 | 14.6810 | 0.0341 | 0.0706 | 0.0054 | 0.0171 |
| `adder_n64.qasm` | 64 | 0.3713 | 11.1483 | 0.1426 | 0.5736 | 0.0399 | 0.2903 |
| `swap_test_n41.qasm` | 41 | 0.9555 | 9.8554 | 0.0647 | 0.2828 | 0.0208 | 0.0545 |
| `swap_test_n83.qasm` | 83 | 0.5975 | 12.1070 | 0.3040 | 5.2608 | 0.0476 | 0.4462 |
| `knn_n41.qasm` | 41 | 0.9797 | 9.9287 | 0.0643 | 0.2816 | 0.0210 | 0.0530 |
| `knn_n67.qasm` | 67 | 0.2663 | 2.4026 | 0.1494 | 1.5224 | 0.0418 | 0.3800 |